# English LLM Benchmark Suite

> **Based on:** [hyogrin/rhoai-lmeval-builder-lab](https://github.com/hyogrin/rhoai-lmeval-builder-lab) (EvalHub SDK pattern)  
> **Benchmarks:** MMLU, ARC Challenge, HellaSwag, Winogrande, TruthfulQA, GSM8K  
> **Adapted by:** [gymnatics/RHOAI-Toolkit](https://github.com/gymnatics/RHOAI-Toolkit)

In [ ]:
!pip install -q "eval-hub-sdk[client]"

In [ ]:
# Auto-configure environment for this cluster
# Env vars are injected via notebook-env ConfigMap (created by deploy.sh)
# Fallback: tries .env file or oc auto-detection for local/standalone use
import os
import subprocess

try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env"), override=True)
except ImportError:
    pass

NAMESPACE = os.environ.get("NAMESPACE", "lmeval-demo")
MODEL_NAME = os.environ.get("MODEL_NAME", "")
MODEL_NAMESPACE = os.environ.get("MODEL_NAMESPACE", "")
BASE_URL = os.environ.get("BASE_URL", "")
EVALHUB_URL = os.environ.get("EVALHUB_URL", "https://evalhub.redhat-ods-applications.svc:8443")

# Fallback auto-detection (for running outside a workbench)
if not MODEL_NAME or not BASE_URL:
    detected = ""
    detected_ns = ""
    try:
        result = subprocess.run(
            ["oc", "get", "llminferenceservice", "-A", "--no-headers",
             "-o", "custom-columns=NS:.metadata.namespace,NAME:.metadata.name"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0 and result.stdout.strip():
            parts = result.stdout.strip().split("\n")[0].split()
            detected_ns, detected = parts[0], parts[1]
    except Exception:
        pass

    if detected:
        MODEL_NAME = MODEL_NAME or detected
        MODEL_NAMESPACE = MODEL_NAMESPACE or detected_ns
        if not BASE_URL:
            BASE_URL = f"https://{detected}-kserve-workload-svc.{detected_ns}.svc:8000/v1"

MODEL_NAMESPACE = MODEL_NAMESPACE or NAMESPACE
print(f"Namespace:       {NAMESPACE}")
print(f"Model:           {MODEL_NAME}")
print(f"Model Namespace: {MODEL_NAMESPACE}")
print(f"Base URL:        {BASE_URL}")
print(f"EvalHub:         {EVALHUB_URL}")

if not MODEL_NAME:
    print("\n⚠ MODEL_NAME not set. Either:")
    print("  - Run: scripts/refresh-notebook-env.sh lmeval-demo")
    print("  - Or set MODEL_NAME manually in this cell")

## Step 1: Configuration

In [ ]:
import subprocess

# Auth token: prefer injected env var (evalhub-service SA token), fall back to oc whoami -t
EVALHUB_AUTH_TOKEN = os.environ.get("EVALHUB_AUTH_TOKEN", "")
if not EVALHUB_AUTH_TOKEN:
    _r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    EVALHUB_AUTH_TOKEN = _r.stdout.strip() if _r.returncode == 0 else None
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "https://mlflow.redhat-ods-applications.svc:8443")
LIMIT = int(os.environ.get("LIMIT", "5"))

print(f"Namespace:       {NAMESPACE}")
print(f"Model Name:      {MODEL_NAME}")
print(f"Model Endpoint:  {BASE_URL}")
print(f"EvalHub URL:     {EVALHUB_URL}")
print(f"MLflow URI:      {MLFLOW_TRACKING_URI}")
print(f"Sample Limit:    {LIMIT}")

## Step 2: Initialize the EvalHub Client

In [ ]:
from evalhub import (
    SyncEvalHubClient,
    ModelConfig,
    BenchmarkConfig,
    JobSubmissionRequest,
    ExperimentConfig,
    ExperimentTag,
    JobStatus,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

model = ModelConfig(
    url=BASE_URL,
    name=MODEL_NAME,
)

print(f"EvalHub client connected: {EVALHUB_URL}")
print(f"Model target:             {model.name} @ {model.url}")

## Step 3: Discover Available English Benchmarks

Query EvalHub for standard English benchmarks (MMLU, ARC, HellaSwag, Winogrande, TruthfulQA, GSM8K).

In [ ]:
# Standard English benchmarks (lighteval provider IDs)
ENGLISH_BENCHMARKS = [
    "mmlu",              # Massive Multitask Language Understanding (57 subjects)
    "arc:challenge",     # AI2 Reasoning Challenge
    "hellaswag",         # Commonsense NLI
    "winogrande",        # Winograd Schema Challenge
    "truthfulqa:mc",     # TruthfulQA (multiple choice)
    "gsm8k",             # Grade School Math 8K
]

all_benchmarks = client.benchmarks.list(provider_id="lighteval")
all_benchmark_ids = {bm.id.lower() for bm in all_benchmarks}

english_benchmarks = [
    bm for bm in all_benchmarks
    if bm.id.lower() in [b.lower() for b in ENGLISH_BENCHMARKS]
]

# Also find any others that match common English eval keywords
english_keywords = ["mmlu", "arc", "hellaswag", "winogrande", "truthful", "gsm", "ifeval", "bbh"]
extra = [
    bm for bm in all_benchmarks
    if any(kw in bm.id.lower() for kw in english_keywords) and bm not in english_benchmarks
]
english_benchmarks.extend(extra)

print(f"Total benchmarks available: {len(all_benchmarks)}")
print(f"English benchmarks found:   {len(english_benchmarks)}")
print("=" * 70)
for bm in english_benchmarks:
    metrics_str = ", ".join(bm.metrics[:3]) if bm.metrics else "N/A"
    print(f"  {bm.id:40s}  metrics=[{metrics_str}]")

if not english_benchmarks:
    print("\nNo pre-registered benchmarks found. Using built-in lm_evaluation_harness tasks directly.")

---

## Evaluation 1: Single English Benchmark (MMLU)

Start with MMLU to verify the pipeline works end-to-end.

### 1-A: Submit the Job

In [ ]:
single_request = JobSubmissionRequest(
    name="mmlu-eval",
    description="MMLU benchmark - 57 subjects, 5-shot",
    tags=["english", "mmlu", "knowledge"],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="mmlu",
            provider_id="lighteval",
            parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT, "num_fewshot": 5},
        ),
    ],
    experiment=ExperimentConfig(
        name="english-mmlu-eval",
        tags=[
            ExperimentTag(key="language", value="english"),
            ExperimentTag(key="benchmark_suite", value="mmlu"),
            ExperimentTag(key="evaluation_type", value="knowledge"),
        ],
    ),
)

job = client.jobs.submit(single_request)

print(f"Job submitted successfully!")
print(f"  Job ID:        {job.id}")
print(f"  Name:          {job.name}")
print(f"  State:         {job.state.value}")
print(f"  MLflow Exp ID: {job.resource.mlflow_experiment_id or 'pending'}")

### 1-B: Monitor Progress

In [ ]:
import time
from datetime import datetime

TERMINAL_STATES = {
    JobStatus.COMPLETED,
    JobStatus.FAILED,
    JobStatus.CANCELLED,
}


def wait_for_job(client, job_id, poll_interval=10, max_wait=600):
    """Poll job status until terminal state or timeout."""
    start = time.time()
    print(f"Monitoring job {job_id}...")
    print("-" * 70)

    while time.time() - start < max_wait:
        status = client.jobs.get(job_id)
        state = status.effective_state
        elapsed = int(time.time() - start)

        msg = ""
        if status.status and status.status.message:
            msg = f" | {status.status.message.message}"

        bm_info = ""
        if status.status and status.status.benchmarks:
            bm_states = [f"{b.id}={b.state.value}" for b in status.status.benchmarks]
            bm_info = f" | benchmarks: {', '.join(bm_states)}"

        print(f"  [{elapsed:>4d}s] {state.value:>16s}{msg}{bm_info}")

        if state in TERMINAL_STATES:
            break

        time.sleep(poll_interval)

    print("-" * 70)
    print(f"Final state: {state.value} (elapsed: {elapsed}s)")
    return status


completed_job = wait_for_job(client, job.id)

### 1-C: View Results

In [ ]:
def display_job_results(job):
    """Display evaluation results with MLflow links."""
    if not job.results:
        print("No results available.")
        return

    print("Evaluation Results")
    print("=" * 70)

    if job.results.mlflow_experiment_url:
        print(f"\n  MLflow Experiment: {job.results.mlflow_experiment_url}")

    for bm in job.results.benchmarks:
        print(f"\n  Benchmark: {bm.id}")
        print(f"  Provider:  {bm.provider_id}")
        if bm.mlflow_run_id:
            print(f"  MLflow Run: {bm.mlflow_run_id}")

        if bm.metrics:
            print(f"  Metrics:")
            for name, value in bm.metrics.items():
                if isinstance(value, float):
                    print(f"    {name:30s} = {value:.4f}")
                else:
                    print(f"    {name:30s} = {value}")
        else:
            print("  Metrics: (none)")


display_job_results(completed_job)

---

## Evaluation 2: Full English Benchmark Suite (Optional)

Run all 6 standard English benchmarks in a single job. This takes longer but gives a comprehensive view of model capabilities.

**Tip:** Adjust `LIMIT` in the config cell to control sample size (e.g., `5` for quick smoke tests, `None` for full evaluation).

In [ ]:
suite_benchmarks = [
    BenchmarkConfig(id="mmlu",            provider_id="lighteval", parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT, "num_fewshot": 5}),
    BenchmarkConfig(id="arc:challenge",   provider_id="lighteval", parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT, "num_fewshot": 25}),
    BenchmarkConfig(id="hellaswag",       provider_id="lighteval", parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT, "num_fewshot": 10}),
    BenchmarkConfig(id="winogrande",      provider_id="lighteval", parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT, "num_fewshot": 5}),
    BenchmarkConfig(id="truthfulqa:mc",   provider_id="lighteval", parameters={"temperature": 0.0, "max_tokens": 16, "limit": LIMIT, "num_fewshot": 0}),
    BenchmarkConfig(id="gsm8k",           provider_id="lighteval", parameters={"temperature": 0.0, "max_tokens": 256, "limit": LIMIT, "num_fewshot": 5}),
]

suite_request = JobSubmissionRequest(
    name=f"english-suite-{MODEL_NAME}",
    description=f"Full English benchmark suite for {MODEL_NAME}",
    tags=["english", "full-suite", MODEL_NAME],
    model=model,
    benchmarks=suite_benchmarks,
    experiment=ExperimentConfig(
        name=f"english-suite-{MODEL_NAME}",
        tags=[
            ExperimentTag(key="language", value="english"),
            ExperimentTag(key="benchmark_suite", value="open-llm-leaderboard"),
            ExperimentTag(key="evaluation_type", value="comprehensive"),
        ],
    ),
)

suite_job = client.jobs.submit(suite_request)

print(f"Suite job submitted!")
print(f"  Job ID:     {suite_job.id}")
print(f"  Name:       {suite_job.name}")
print(f"  Benchmarks: {[b.id for b in suite_benchmarks]}")
print(f"  State:      {suite_job.state.value}")
print(f"\nMonitoring (this may take a while)...")

suite_result = wait_for_job(client, suite_job.id, poll_interval=15, max_wait=1800)
display_job_results(suite_result)

---

## Step 4: Job Management

List, inspect, and manage all evaluation jobs.

### List All Jobs

In [ ]:
jobs_list = client.jobs.list()

print(f"Total Jobs: {len(jobs_list)}")
print("=" * 90)
print(f"{'State':>16s}  {'Job ID':12s}  {'Name':30s}  {'Experiment':25s}  Benchmarks")
print("-" * 90)
for j in jobs_list:
    state = j.effective_state.value
    exp = j.experiment.name if j.experiment else "N/A"
    bms = [b.id for b in j.benchmarks] if j.benchmarks else []
    print(f"{state:>16s}  {j.id[:12]:12s}  {j.name[:30]:30s}  {exp[:25]:25s}  {bms}")

### Inspect a Specific Job

Replace `JOB_ID` with the job you want to inspect.

In [ ]:
# Replace with actual job ID:
# JOB_ID = "your-job-id-here"
# inspected = client.jobs.get(JOB_ID)
# display_job_results(inspected)

---

## Step 5: Export Results

### Export to Markdown

In [ ]:
def collect_results_table(client):
    """Collect benchmark metrics from completed jobs into a comparison dict."""
    jobs_list = client.jobs.list()
    table = {}
    for j in jobs_list:
        if j.effective_state != JobStatus.COMPLETED or not j.results:
            continue
        for bm in j.results.benchmarks:
            if bm.id not in table:
                table[bm.id] = {}
            table[bm.id][j.name] = bm.metrics
    return table


comparison = collect_results_table(client)


def results_to_markdown(comparison, title="EvalHub Benchmark Results"):
    """Convert comparison results to markdown table."""
    if not comparison:
        return "No results available."

    lines = [f"## {title}", ""]

    for benchmark_id, job_results in comparison.items():
        lines.append(f"### {benchmark_id}")
        lines.append("")

        all_metrics = set()
        for metrics in job_results.values():
            all_metrics.update(metrics.keys())
        all_metrics = sorted(all_metrics)

        job_names = sorted(job_results.keys())
        header = "| Metric | " + " | ".join(job_names) + " |"
        sep = "|---" + "|---" * len(job_names) + "|"
        lines.extend([header, sep])

        for metric in all_metrics:
            row = f"| {metric} |"
            for job_name in job_names:
                val = job_results.get(job_name, {}).get(metric, "-")
                if isinstance(val, float):
                    row += f" {val:.4f} |"
                else:
                    row += f" {val} |"
            lines.append(row)

        lines.append("")

    return "\n".join(lines)


md = results_to_markdown(comparison)
print(md)

### Save Results to JSON

In [ ]:
import json
from pathlib import Path

results_root = Path("../results")

jobs_list = client.jobs.list()
saved = 0
for j in jobs_list:
    if j.effective_state != JobStatus.COMPLETED or not j.results:
        continue

    model_name = j.model.name if j.model else "unknown"
    model_dir = results_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    job_data = {
        "job_id": j.id,
        "name": j.name,
        "model": {"url": j.model.url, "name": j.model.name},
        "experiment": j.experiment.name if j.experiment else None,
        "benchmarks": [
            {
                "id": bm.id,
                "provider_id": bm.provider_id,
                "metrics": bm.metrics,
                "mlflow_run_id": bm.mlflow_run_id,
            }
            for bm in j.results.benchmarks
        ],
    }

    filename = f"{j.name}_{j.id[:8]}.json"
    output_path = model_dir / filename
    with open(output_path, "w") as f:
        json.dump(job_data, f, indent=2, default=str)
    print(f"Saved: {output_path}")
    saved += 1

print(f"\n{saved} result(s) saved to {results_root.resolve()}/<model-name>/")

---

## Summary

This notebook demonstrated English LLM benchmarking via EvalHub:

1. **Configuration** and EvalHub client setup with auto-detected model
2. **Single benchmark** evaluation (MMLU) with MLflow tracking
3. **Job management** -- list and inspect jobs
4. **Export** -- Markdown and JSON output

### Available Benchmarks

| Benchmark | What it Tests | Few-shot |
|-----------|--------------|----------|
| MMLU | Knowledge across 57 subjects | 5-shot |
| ARC Challenge | Science reasoning | 25-shot |
| HellaSwag | Commonsense NLI | 10-shot |
| Winogrande | Coreference resolution | 5-shot |
| TruthfulQA MC2 | Truthfulness | 0-shot |
| GSM8K | Math word problems | 5-shot |

To run additional benchmarks, modify the `BenchmarkConfig(id=...)` in Step 1-A.

### Next Steps

- **guidellm-benchmark.ipynb** -- GuideLLM throughput/latency profiling
- **korean-mcq-benchmark.ipynb** -- Korean language evaluation
